# 🕷️ Web Scraping & Web Crawling — Complete Guide

**What you will learn:**
- `requests` + `BeautifulSoup` → scrape static HTML pages
- `Selenium` → scrape dynamic JavaScript pages
- `Scrapy` → crawl across many pages automatically

**Practice site used:** [quotes.toscrape.com](https://quotes.toscrape.com) — safe, legal, made for learning

---
**Run order:** Execute cells top to bottom. Each section is independent.


## 📦 Section 0 — Install Libraries
Run this cell first. Only needs to run once per Colab session.

In [ ]:
import os
os.system("pkill chrome")
os.system("pkill chromedriver")
print("✅ Cleaned up old processes")

✅ Cleaned up old processes


In [ ]:
!pip install requests beautifulsoup4 selenium webdriver-manager scrapy -q
print('✅ All libraries installed!')

✅ All libraries installed!


---
## 📄 Section 1 — Static Scraping: `requests` + `BeautifulSoup`

**Use this when:**
- Page content is visible in browser's View Page Source
- No login or JavaScript is needed
- You want a fast, lightweight scrape

**How it works:**
```
requests.get(url)  →  raw HTML text
BeautifulSoup(html)  →  searchable tree
soup.find_all()  →  extracted data
```

### Step 1 — Import libraries and fetch the page

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import csv

URL = 'https://quotes.toscrape.com/'

# User-Agent header makes the request look like a real browser
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
    )
}

try:
    response = requests.get(URL, headers=headers, timeout=10)
    response.raise_for_status()  # raises error for 4xx/5xx status codes
    print(f'✅ Page fetched! Status: {response.status_code}')
except requests.exceptions.RequestException as e:
    print(f'❌ Failed: {e}')
    response = None

✅ Page fetched! Status: 200


### Step 2 — Parse HTML with BeautifulSoup

In [ ]:
if response:
    # html.parser is Python's built-in parser — no extra install needed
    soup = BeautifulSoup(response.text, 'html.parser')

    print(f'📄 Page title: {soup.title.get_text()}')
    print(f'\n🔍 First 300 chars of body text:')
    print(soup.body.get_text(strip=True)[:300])

📄 Page title: Quotes to Scrape

🔍 First 300 chars of body text:
Quotes to ScrapeLogin“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”byAlbert Einstein(about)Tags:changedeep-thoughtsthinkingworld“It is our choices, Harry, that show what we truly are, far more than our abilities.”byJ.K. Rowling(abo


### Step 3 — Find elements

| Method | What it does |
|---|---|
| `soup.find('tag')` | First matching element |
| `soup.find_all('tag', class_='name')` | All matching elements |
| `soup.select('css selector')` | CSS selector (like JavaScript) |
| `.get_text(strip=True)` | Extract visible text |
| `tag['href']` | Read an attribute value |

In [ ]:
if response:
    quotes_data = []
    quote_blocks = soup.find_all('div', class_='quote')
    print(f'🔎 Found {len(quote_blocks)} quotes on page 1:\n')

    for block in quote_blocks:
        text   = block.find('span', class_='text').get_text(strip=True)
        author = block.find('small', class_='author').get_text(strip=True)
        tags   = [t.get_text(strip=True) for t in block.find_all('a', class_='tag')]

        quotes_data.append({'text': text, 'author': author, 'tags': tags})

        print(f'  Quote : {text[:65]}...')
        print(f'  Author: {author}')
        print(f'  Tags  : {", ".join(tags)}')
        print()

🔎 Found 10 quotes on page 1:

  Quote : “The world as we have created it is a process of our thinking. It...
  Author: Albert Einstein
  Tags  : change, deep-thoughts, thinking, world

  Quote : “It is our choices, Harry, that show what we truly are, far more ...
  Author: J.K. Rowling
  Tags  : abilities, choices

  Quote : “There are only two ways to live your life. One is as though noth...
  Author: Albert Einstein
  Tags  : inspirational, life, live, miracle, miracles

  Quote : “The person, be it gentleman or lady, who has not pleasure in a g...
  Author: Jane Austen
  Tags  : aliteracy, books, classic, humor

  Quote : “Imperfection is beauty, madness is genius and it's better to be ...
  Author: Marilyn Monroe
  Tags  : be-yourself, inspirational

  Quote : “Try not to become a man of success. Rather become a man of value...
  Author: Albert Einstein
  Tags  : adulthood, success, value

  Quote : “It is better to be hated for what you are than to be loved for w...
  Author: Andr

### Step 4 — Save extracted data to JSON and CSV

In [ ]:
if quotes_data:
    # Save to JSON — preserves the tags list as a real list
    with open('quotes_static.json', 'w', encoding='utf-8') as f:
        json.dump(quotes_data, f, indent=2, ensure_ascii=False)
    print('💾 Saved to quotes_static.json')

    # Save to CSV — flatten tags to a comma-separated string
    with open('quotes_static.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['text', 'author', 'tags'])
        writer.writeheader()
        for q in quotes_data:
            writer.writerow({**q, 'tags': ', '.join(q['tags'])})
    print('💾 Saved to quotes_static.csv')

    # Preview the JSON
    print('\n📋 Preview (first 2 items):')
    print(json.dumps(quotes_data[:2], indent=2))

💾 Saved to quotes_static.json
💾 Saved to quotes_static.csv

📋 Preview (first 2 items):
[
  {
    "text": "\u201cThe world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.\u201d",
    "author": "Albert Einstein",
    "tags": [
      "change",
      "deep-thoughts",
      "thinking",
      "world"
    ]
  },
  {
    "text": "\u201cIt is our choices, Harry, that show what we truly are, far more than our abilities.\u201d",
    "author": "J.K. Rowling",
    "tags": [
      "abilities",
      "choices"
    ]
  }
]


---
## 🌐 Section 2 — Dynamic Scraping: `Selenium`

**Use this when:**
- Content loads via JavaScript (page looks empty in View Source)
- You need to click buttons, fill forms, scroll, or log in
- `requests` returns incomplete or empty HTML

**How it works:**
```
Selenium launches a real Chrome browser (hidden)
→ loads the page fully including all JavaScript
→ you interact with it (click, type, scroll)
→ hand page_source to BeautifulSoup to parse
```

> ⚠️ **Always call `driver.quit()` when done** — or Chrome keeps running and wastes memory.

### Step 1 — Setup and launch the browser

In [ ]:
import os, shutil, time

# ── Step 1: Install Chromium + matching ChromeDriver ──────────────────────────
print('Installing Chrome...')
os.system('apt-get update -q')
os.system('apt-get install -y -q chromium-browser chromium-chromedriver')
print('✅ Chrome installed!')

# ── Step 2: Kill leftover processes ───────────────────────────────────────────
os.system('pkill -f chromium 2>/dev/null')
os.system('pkill -f chromedriver 2>/dev/null')
time.sleep(1)
print('✅ Cleaned up old processes')

# ── Step 3: Find exact binary paths ───────────────────────────────────────────
chrome_path = shutil.which('chromium-browser') or shutil.which('chromium')
driver_path = (
    shutil.which('chromedriver')
    or '/usr/lib/chromium-browser/chromedriver'
    or '/usr/bin/chromedriver'
)

if chrome_path:
    print(f'✅ Chromium found at: {chrome_path}')
else:
    raise RuntimeError('❌ Chromium not found. Re-run the install step above.')

# Confirm versions match — both should show the same number (e.g. 130.x)
os.system(f'{chrome_path} --version')
os.system(f'{driver_path} --version')

# ── Step 4: Imports ───────────────────────────────────────────────────────────
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

# ── Step 5: Configure Chrome options ──────────────────────────────────────────
# Every one of these flags is required for Colab's sandboxed environment
options = Options()
options.binary_location = chrome_path          # point to Chromium, not google-chrome
options.add_argument('--headless=new')         # invisible browser
options.add_argument('--no-sandbox')           # required in Colab
options.add_argument('--disable-dev-shm-usage')# prevents shared memory crashes
options.add_argument('--disable-gpu')
options.add_argument('--no-zygote')            # disables Chrome process spawner
options.add_argument('--single-process')       # required in Colab sandbox
options.add_argument('--disable-setuid-sandbox')
options.add_argument('--disable-software-rasterizer')
options.add_argument('--disable-background-networking')
options.add_argument('--disable-default-apps')
options.add_argument('--disable-sync')
options.add_argument('--no-first-run')
options.add_argument('--remote-debugging-pipe') # more stable than port in Colab
options.add_argument('--window-size=1920,1080')
options.add_argument(
    'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
    'AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
)

# ── Step 6: Launch browser using SYSTEM chromedriver ──────────────────────────
# ⚠️ Do NOT use ChromeDriverManager() — it downloads a mismatched version
#    which causes the DevToolsActivePort crash you saw earlier
try:
    driver = webdriver.Chrome(
        service=Service(executable_path=driver_path),
        options=options
    )
    print(f'✅ Browser launched! Version: {driver.capabilities["browserVersion"]}')

except Exception as e:
    print(f'❌ Launch failed: {e}')
    print('\n⚡ Trying fallback with undetected-chromedriver...')
    os.system('pip install undetected-chromedriver -q')
    import undetected_chromedriver as uc
    uc_options = uc.ChromeOptions()
    uc_options.add_argument('--headless=new')
    uc_options.add_argument('--no-sandbox')
    uc_options.add_argument('--disable-dev-shm-usage')
    uc_options.add_argument('--no-zygote')
    uc_options.add_argument('--single-process')
    driver = uc.Chrome(
        driver_executable_path=driver_path,
        browser_executable_path=chrome_path,
        options=uc_options,
        use_subprocess=False
    )
    print('✅ Browser launched via fallback!')

# ── Quick test ────────────────────────────────────────────────────────────────
driver.get('https://quotes.toscrape.com/')
print(f'📄 Test page title: {driver.title}')
print('🎉 Selenium is working correctly!')

Installing Chrome...
✅ Chrome installed!
✅ Cleaned up old processes
✅ Chromium found at: /usr/bin/chromium-browser
❌ Launch failed: Message: Service /usr/bin/chromedriver unexpectedly exited. Status code was: 1


⚡ Trying fallback with undetected-chromedriver...


WebDriverException: Message: Service /usr/bin/chromedriver unexpectedly exited. Status code was: 1


### Step 2 — Open a page and find elements

In [ ]:
try:
    driver.get('https://quotes.toscrape.com/')
    print(f'📄 Title : {driver.title}')
    print(f'🌐 URL   : {driver.current_url}')
except Exception as e:
    print(f'❌ Failed to open page: {e}')

# find_elements returns a list of matching elements
# Common locators:
#   By.CLASS_NAME  →  finds elements by CSS class
#   By.ID          →  finds elements by id attribute
#   By.TAG_NAME    →  finds elements by HTML tag
#   By.CSS_SELECTOR → full CSS selector (most powerful)
#   By.XPATH       →  XPath selector (fallback for complex cases)

quotes = driver.find_elements(By.CLASS_NAME, 'quote')
print(f'\n🔎 Found {len(quotes)} quote blocks via Selenium')

print('\nFirst 3 quotes (raw text):')
for q in quotes[:3]:
    print(q.text[:80])
    print('---')

❌ Failed to open page: name 'driver' is not defined


NameError: name 'driver' is not defined

### Step 3 — Interact with the page (click, type, wait)

**`WebDriverWait`** pauses your script until an element is ready.  
Always use this instead of `time.sleep()` — it's faster and more reliable.

In [ ]:
driver.get('https://duckduckgo.com')

try:
    # Wait up to 10 seconds for the search box to appear
    # FIX: updated selector — DuckDuckGo's old By.NAME 'q' selector is broken
    search_box = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "input[type='text']"))
    )
    print(f'✅ Search box found on: {driver.title}')

    search_box.clear()
    search_box.send_keys('Python web scraping tutorial')
    search_box.send_keys(Keys.RETURN)  # press Enter

    # Wait for results to load before reading the page
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'article, [data-testid]'))
    )
    print(f'✅ Results loaded. Title: {driver.title}')

except Exception as e:
    print(f'❌ Interaction failed: {e}')
    # Save a screenshot for debugging — very useful when things break!
    driver.save_screenshot('debug_screenshot.png')
    print('📸 Screenshot saved: debug_screenshot.png')

### Step 4 — Parse Selenium's output with BeautifulSoup

In [ ]:
# driver.page_source gives you the full rendered HTML (after JavaScript ran)
# Hand it to BeautifulSoup for familiar, easy parsing

soup = BeautifulSoup(driver.page_source, 'html.parser')
headings = soup.find_all('h2')

print('🔎 Result headings found:')
for i, h in enumerate(headings[:5], 1):
    text = h.get_text(strip=True)
    if text:
        print(f'  {i}. {text}')

### Step 5 — Always close the browser when done

In [ ]:
# FIX: driver.quit() was missing in the original file
# Without this, Chrome keeps running in the background and wastes memory

driver.quit()
print('✅ Browser closed cleanly.')

---
## 🕷️ Section 3 — Web Crawling: `Scrapy`

**Use this when:**
- You need to scrape many pages (not just one)
- You want to follow "Next page" links automatically
- You need fast, parallel, production-grade crawling

**How a Scrapy spider works:**
```
start_urls  →  parse()  →  yield item (save data)
                        →  yield response.follow() (go to next page)
                                  ↑_______________________________|
                                  repeats until no more pages
```

> ⚠️ **Colab issue:** Scrapy has its own event loop which conflicts with Jupyter.  
> **Fix:** Write the spider to a `.py` file and run it with `!scrapy runspider` (shell command).

### Step 1 — Write the spider to a file

In [ ]:
spider_code = '''
import scrapy

class QuotesSpider(scrapy.Spider):
    """
    Crawls ALL pages of quotes.toscrape.com automatically.

    Key concepts:
      - start_urls: where the spider begins
      - parse(): called on every page visited
      - yield item: sends extracted data to output
      - response.follow(): queues the next page to visit
    """
    name = "quotes"
    custom_settings = {
        "DOWNLOAD_DELAY": 1,         # wait 1 second between pages (polite!)
        "CONCURRENT_REQUESTS": 1,    # one request at a time
        "LOG_LEVEL": "WARNING",      # cleaner output
    }
    start_urls = ["https://quotes.toscrape.com/"]

    def parse(self, response):
        # Scrapy uses CSS selectors directly — no BeautifulSoup needed
        # ::text  → extracts the text content of an element
        # .get()  → first result (or None)
        # .getall() → all results as a list

        for quote in response.css("div.quote"):
            yield {
                "text":   quote.css("span.text::text").get(),
                "author": quote.css("small.author::text").get(),
                "tags":   quote.css("a.tag::text").getall(),
            }

        # Follow the Next button link — this is the crawling part
        next_page = response.css("li.next a::attr(href)").get()
        if next_page:
            yield response.follow(next_page, callback=self.parse)
        else:
            print("  Crawl complete — no more pages!")
'''

with open('quotes_spider.py', 'w', encoding='utf-8') as f:
    f.write(spider_code)

print('✅ Spider file written: quotes_spider.py')

✅ Spider file written: quotes_spider.py


### Step 2 — Run the spider

In [ ]:
# -o quotes_crawled.json  → saves all yielded items to a JSON file
# Scrapy will automatically follow all Next page links and stop when done

!scrapy runspider quotes_spider.py -o quotes_crawled.json
print('\n✅ Crawl complete!')

2026-03-24 09:11:15 [scrapy.utils.log] INFO: Scrapy 2.14.2 started (bot: scrapybot)
2026-03-24 09:11:15 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.0.2',
 'libxml2': '2.14.6',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.4.1',
 'Twisted': '25.5.0',
 'Python': '3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]',
 'pyOpenSSL': '24.2.1 (OpenSSL 3.3.2 3 Sep 2024)',
 'cryptography': '43.0.3',
 'Platform': 'Linux-6.6.113+-x86_64-with-glibc2.35'}
2026-03-24 09:11:15 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-03-24 09:11:15 [asyncio] DEBUG: Using selector: EpollSelector
  Crawl complete — no more pages!

✅ Crawl complete!


### Step 3 — Load and inspect the crawled results

In [ ]:
import json

with open('quotes_crawled.json', 'r', encoding='utf-8') as f:
    crawled = json.load(f)

print(f'📊 Total quotes crawled across ALL pages: {len(crawled)}')
print('\nFirst 3 results:')
for item in crawled[:3]:
    print(f'  Author: {item["author"]}')
    print(f'  Quote : {item["text"][:70]}...')
    print(f'  Tags  : {", ".join(item["tags"])}')
    print()

📊 Total quotes crawled across ALL pages: 100

First 3 results:
  Author: Albert Einstein
  Quote : “The world as we have created it is a process of our thinking. It cann...
  Tags  : change, deep-thoughts, thinking, world

  Author: J.K. Rowling
  Quote : “It is our choices, Harry, that show what we truly are, far more than ...
  Tags  : abilities, choices

  Author: Albert Einstein
  Quote : “There are only two ways to live your life. One is as though nothing i...
  Tags  : inspirational, life, live, miracle, miracles



---
## 📋 Section 4 — Quick Reference Cheat Sheet


### When to use which tool

| Situation | Best tool |
|---|---|
| Static HTML, no JavaScript | `requests` + `BeautifulSoup` |
| Page loads content via JavaScript | `Selenium` |
| Need to click / type / scroll | `Selenium` |
| Crawl 10+ pages automatically | `Scrapy` |
| Quick one-off extraction | `requests` + `BeautifulSoup` |
| Production scraper at scale | `Scrapy` |

---

### BeautifulSoup selector cheat sheet

```python
soup.find('div')                          # first <div>
soup.find('div', class_='quote')          # first <div class='quote'>
soup.find('a', id='next')                 # first <a id='next'>
soup.find_all('p')                        # all <p> tags
soup.select('div.quote span.text')        # CSS selector
soup.select_one('li.next a')['href']      # read attribute
tag.get_text(strip=True)                  # clean text content
```

---

### Selenium locator cheat sheet

```python
By.ID              # find by id attribute
By.CLASS_NAME      # find by CSS class
By.TAG_NAME        # find by HTML tag
By.CSS_SELECTOR    # full CSS selector (most powerful)
By.XPATH           # XPath expression (best for complex cases)
By.LINK_TEXT       # find link by its visible text
```

---

### Scrapy CSS selector cheat sheet

```python
response.css('div.quote')                 # all matching elements
response.css('span.text::text').get()     # text of first match
response.css('a.tag::text').getall()      # text of all matches
response.css('a::attr(href)').get()       # attribute value
response.follow(url, callback=self.parse) # crawl next page
```

---

### Bugs fixed from original file

| # | Bug | Fix |
|---|---|---|
| 1 | Missing `WebDriverWait` import | Added import |
| 2 | Missing `expected_conditions` import | Added import |
| 3 | DuckDuckGo `By.NAME 'q'` selector broken | Changed to `input[type='text']` |
| 4 | `driver.quit()` missing | Added at end of Selenium section |
| 5 | No error handling on requests | Added `try/except` blocks |
| 6 | No `DOWNLOAD_DELAY` in Scrapy | Added `custom_settings` |
